# Image Processing: MSE/CHE 7530/5995, F2026
Purpose: These exercises will give you an opportunity to engage with the material presented in lecture in a hands-on way. Concepts that I recommend you absorb for the exam will be bolded and explicitly hinted to you.  
A full disclaimer: I am not a professional at this and may sometimes do things inefficiently, or there would have been a better way to process the image for the stated goal. The primary goal is for you to know how basic image processing principles work.

## Standard Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Image Processing & AFM Reading Packages

In [ ]:
import skimage as ski
import scipy.ndimage as ndi
import cv2

In [ ]:
!pip install uv -q
!uv pip install pySPM --system -q

import pySPM as spm

In [ ]:
!rm -rf /content/MSE7530/
! git clone https://github.com/cbishop4/MSE7530

## Image Smoothing and Filtering  
Here, we will use image filtering to subtract background fluctuations from an image. We will use two filters: a **Gaussian** and a **mean ("uniform") filter**. **You should know what these two filters do and how they differ**.

#### Mean vs. Gaussian filters  
First we will look at how these two filters work. A **mean** filter will average all pixels in the neighborhood (defined by the parameter "size") and replace the pixel with that value. A **Gaussian** filter will do something similar, except it will average pixels over a Gaussian distribution; this will give greater weight to the pixels closer to the one being filtered. The "sigma" parameter influences how wide the Gaussian filter extends.

To explicitly illustrate, I will generate a synthetic image.

In [ ]:
filt_ex = np.random.rand(20,20)*5
plt.imshow(filt_ex)
plt.colorbar()

In [ ]:
fig, ax = plt.subplots(1,3)
gauss_filtered = ndi.gaussian_filter(filt_ex, sigma = 1)
mean_filtered = ndi.uniform_filter(filt_ex, size=2)
cb = ax[0].imshow(filt_ex)
plt.colorbar(cb,ax=ax[0],location='bottom')
cb = ax[1].imshow(mean_filtered)
plt.colorbar(cb, ax=ax[1],location='bottom')
cb = ax[2].imshow(gauss_filtered)
plt.colorbar(cb, ax=ax[2],location='bottom')

Exercise: play with the size of the array and the two filter sizes to convince yourself of how the mean and Gaussian Filters differ. Below is a function:

In [ ]:
def experiment_filters(box_dim, gauss_sigma, mean_size):
  fig, ax = plt.subplots(1,4,figsize=(20,4))
  filt_ex = np.random.rand(box_dim,box_dim)
  gauss_filtered = ndi.gaussian_filter(filt_ex, sigma = gauss_sigma)
  mean_filtered = ndi.uniform_filter(filt_ex, size=mean_size)
  cb = ax[0].imshow(filt_ex)
  plt.colorbar(cb,ax=ax[0],location='bottom')
  cb = ax[1].imshow(mean_filtered)
  plt.colorbar(cb, ax=ax[1],location='bottom')
  cb = ax[2].imshow(gauss_filtered)
  plt.colorbar(cb, ax=ax[2],location='bottom')
  ax[3].plot(filt_ex[:,0],'-o',label='original')
  ax[3].plot(mean_filtered[:,0],'-^',label='Mean')
  ax[3].plot(gauss_filtered[:,0],'-s',label='Gaussian')
  ax[3].legend(); ax[3].set_xlabel('Pixel')
  ax[0].set_title('Original'); ax[1].set_title('Mean filter'); ax[2].set_title('Gaussian Filter')
  return(filt_ex, mean_filtered, gauss_filtered)


A 1-dimensional cut is included in the 4th graph for illustrative purposes; realize that it does not consider the 2nd dimension's influence on the average values of each pixel, so this will not work out exactly quantitatively.

Modify the values for box_dim, gauss_sigma, and mean_size to illustrate the principles for yourself. Note that since the array is generated by a random number generator, you will get a different result each time you run it. You can index specific pixels to find their value in the original, and after filtering, as well as finding the values of the ones next to them:

In [ ]:
og, mf, gf = experiment_filters(box_dim = 10,gauss_sigma=2,mean_size=3)

#### Filtering a real image  
Here you will experiment with the filters on a real image.

In [ ]:
rawAFM = '/content/MSE7530/sampledata/AFMs/310K_2'
scan = spm.Bruker(rawAFM)
ht_chan = scan.get_channel('Height Sensor')
ht_px_vals = ht_chan.pixels
phz_chan = scan.get_channel('Phase')
phz_px_vals = phz_chan.pixels

In [ ]:
# cutting to a smaller portion
img = ht_px_vals[100:200,100:200]
plt.imshow(img)

### Mean Filter
*(Modify sizes to look at effects)*

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(img)
fig.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('Original Image')
mean_filtered = ndi.uniform_filter(img, size=5)
cb = ax[1].imshow(mean_filtered)
fig.colorbar(cb,ax=ax[1],location='bottom')
ax[1].set_title('Mean filter - 5x5 pixel')
mean_filtered = ndi.uniform_filter(img, size=10)
cb = ax[2].imshow(mean_filtered)
fig.colorbar(cb,ax=ax[2],location='bottom')
ax[2].set_title('Mean filter - 10x10 pixel')

### Gaussian Filter

Next, we will use a Gaussian filter. While this is quite similar to a Mean Filter, it gives greater weights to nearer pixels according to a Gaussian distribution. By specifying "sigma", we specify how wide that Gaussian is.

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(img)
fig.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('Original Image')
gauss_filtered = ndi.gaussian_filter(img, sigma=2.5)
cb = ax[1].imshow(gauss_filtered)
fig.colorbar(cb,ax=ax[1],location='bottom')
ax[1].set_title('Gaussian Filter - sigma = 2.5')
gauss_filtered = ndi.gaussian_filter(img, sigma=5)
cb = ax[2].imshow(gauss_filtered)
fig.colorbar(cb,ax=ax[2],location='bottom')
ax[2].set_title('Gaussian Filter - sigma = 5')

Application: We will use a Gaussian filter to eliminate large background fluctuations and focus on the small local features.

In [ ]:
fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(ht_px_vals)
fig.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('Original Image')
gauss_filtered = ndi.gaussian_filter(ht_px_vals, sigma=30)
cb = ax[1].imshow(gauss_filtered)
fig.colorbar(cb,ax=ax[1],location='bottom')
smoothed = ht_px_vals - gauss_filtered
cb = ax[2].imshow(smoothed)
fig.colorbar(cb,ax=ax[2],location='bottom')
ax[2].set_title('Background subtracted')
ax[1].set_title('Smoothed Background')

smoothed = smoothed + np.abs(smoothed.min()) # making values of the smoothed image positive; optional

Note that this may or may not be the optimal sigma to work with. However, for the sake of argument, let's keep this as our image to start the next threshold with.

> **📝 Key Takeaway:** Know the difference between a mean and a Gaussian filter. Understand that you can use a very large sigma value to subtract out background fluctuations in some cases.

## Thresholding: Different Algorithms

First, we can try a few naive thresholds.

In [ ]:
fig, ax = plt.subplots(1,5,figsize=(20,4))
for i in range(5):
  threshed = smoothed > i*2
  ax[i].imshow(threshed)
  ax[i].set_title('Threshold = ' + str(i*2))

We can also use a more sophisticated algorithm to calculate a global threshold. **Be familiar with how and why this global thresholding method works**. You can absolutely use a built-in function for these in an image processing package, but here you'll do it by hand to see how it works. The overall procedure is:  
1. Choose an initial estimate for the threshold $T$  
2. Segment the image to produce a group of pixels $G_1$ with $I>T$, and $G_2$ with $I<T$  
3. Compute the mean intensity values $m_1$ and $m_2$ for the pixels in $G_1$ and $G_2$, respectively  
4. Compute a new threshold by the formula: $T = \frac{1}{2}(m_1 + m_2)$  
5. Repeat until T stops changing (to a pre-defined amount)  
**I recommend that you go through the following code and determine which lines correspond to which of the numbered steps above.**

In [ ]:
# initial estimate: 8 (because we know it's bad)
temp_thresh = smoothed > 8
fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(temp_thresh)
fig.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('Initial guess')
cb = ax[1].imshow(smoothed*temp_thresh)
fig.colorbar(cb,ax=ax[1],location='bottom')
ax[1].set_title('Initial guess - G_1 (I > T)')
cb = ax[2].imshow(smoothed*(~temp_thresh))
fig.colorbar(cb,ax=ax[2],location='bottom')
ax[2].set_title('Initial guess - G_2 (I < T)')

In [ ]:
# compute the means
m_1 = np.mean(smoothed[temp_thresh])
m_2 = np.mean(smoothed[~temp_thresh])
# compute the new threshold
new_thresh = 0.5*(m_1 + m_2)
print(f'{m_1} and {m_2}, so the new threshold is {new_thresh}')

In [ ]:
temp_thresh = smoothed > new_thresh
fig, ax = plt.subplots(1,3,figsize=(15,5))
cb = ax[0].imshow(temp_thresh)
fig.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('2nd iteration')
cb = ax[1].imshow(smoothed*temp_thresh)
fig.colorbar(cb,ax=ax[1],location='bottom')
ax[1].set_title('2nd iteration - G_1 (I > T)')
cb = ax[2].imshow(smoothed*(~temp_thresh))
fig.colorbar(cb,ax=ax[2],location='bottom')
ax[2].set_title('2nd iteration - G_2 (I < T)')

You would then continue iterating until you have converged.

I can also write this up as an automatic loop to find the threshold. This loop will record the threshold over time.

In [ ]:
def find_global_thresh(image, initial_guess):
  """
  Iteratively finds a global intensity threshold using the mean of the two
  pixel groups (above/below threshold) until convergence.
  Returns: (thresh_history, m1s, m2s)
  """
  new_thresh = initial_guess
  thresh_history = [initial_guess]
  m1s = []; m2s = []
  running = True
  while running:
    temp_thresh = image > new_thresh
    m_1 = np.mean(image[temp_thresh])
    m_2 = np.mean(image[~temp_thresh])
    m1s.append(m_1); m2s.append(m_2)
    new_thresh = 0.5*(m_1 + m_2)
    thresh_history.append(new_thresh)
    if abs(new_thresh - thresh_history[-2])/new_thresh < 0.01:
      running = False
  return thresh_history, m1s, m2s


If our initial threshold guess is very low:

In [ ]:
history, m1s, m2s = find_global_thresh(smoothed, 1)
fig, ax = plt.subplots(2,1,figsize=(3,6))
ax[0].plot(history,'o')
ax[1].plot(m1s,'o',label='m1')
ax[1].plot(m2s,'o',label='m2')
ax[0].set_ylabel('Threshold (nm)')
ax[1].set_ylabel('Mean intensity')
ax[1].set_xlabel('Iteration')
ax[1].legend()

And top down:

In [ ]:
history, m1s, m2s = find_global_thresh(smoothed, 10)
fig, ax = plt.subplots(2,1,figsize=(3,6))
ax[0].plot(history,'o')
ax[1].plot(m1s,'o',label='m1')
ax[1].plot(m2s,'o',label='m2')
ax[0].set_ylabel('Threshold (nm)')
ax[1].set_ylabel('Mean intensity')
ax[1].set_xlabel('Iteration')
ax[1].legend()

**Do the two converge to the same global threshold? Know this.**

In [ ]:
fig, ax = plt.subplots(1,3)
ax[0].imshow(smoothed > history[0])
ax[1].imshow(smoothed > history[1])
ax[2].imshow(smoothed > history[-1])

In [ ]:
fig, ax = plt.subplots(1,2)
ax[0].imshow(smoothed)
ax[1].imshow(smoothed*(smoothed > history[-1]))
for a in ax:
  a.set_xlim(0,100); a.set_ylim(0,100)

However, in practice, you would likely use a built-in filter. Above was just for you to learn what it's doing :)

### More sophisticated filters.  
Other filters can be used; these are more sophisticated algorithms that we don't have time to dive into during class (however, do note that we did talk briefly in class about the Otsu filter). Here is an example of a function that will show you the results of multiple filters:

In [ ]:
from skimage.filters import try_all_threshold

fig, ax = try_all_threshold(ht_px_vals, figsize=(10, 10), verbose=False)
plt.show()

In [ ]:
fig, ax = try_all_threshold(smoothed, figsize=(10, 10), verbose=False)
plt.show()

## Distance Transforms  
Distance transforms are used to determine how far each pixel is from the closest "obstacle pixel". They can be used only on binary images. Note that "history[-1]" is the threshold that we determined from the global thresholding method above. **You should understand how to interpret a 2D image of a distance transform**.

In [ ]:
from scipy.ndimage import distance_transform_edt as edt

In [ ]:
matrix_bin = smoothed > history[-1]
holes_bin = smoothed < history[-1]
matrix = smoothed*matrix_bin
holes = smoothed*holes_bin
fig, ax = plt.subplots(1,4,figsize=(20,4))
cb = ax[0].imshow(matrix_bin)
plt.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('Binary Matrix')
cb = ax[1].imshow(matrix)
plt.colorbar(cb,ax=ax[1],location='bottom')
ax[1].set_title('Selected Matrix')
ax[2].imshow(holes_bin)
plt.colorbar(cb,ax=ax[2],location='bottom')
ax[2].set_title('Binary Holes')
cb = ax[3].imshow(holes)
plt.colorbar(cb,ax=ax[3],location='bottom')
ax[3].set_title('Selected Holes')

**What do each of the above images represent?**

Using the binary images, we can use the distance transforms to find the distance of each pixel from the edge.

In [ ]:
transform_holes = edt(holes_bin)
transform_matrix = edt(matrix_bin)
fig, ax = plt.subplots(1,2,figsize=(10,5))
cb = ax[0].imshow(transform_holes)
plt.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('Distance Transform of Holes')
cb = ax[1].imshow(transform_matrix)
plt.colorbar(cb,ax=ax[1],location='bottom')
ax[1].set_title('Distance Transform of Matrix')

Zoomed in to make a little more sense:

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(10,5))
cb = ax[0].imshow(transform_holes)
plt.colorbar(cb,ax=ax[0],location='bottom')
ax[0].set_title('Distance Transform of Holes')
cb = ax[1].imshow(transform_matrix)
plt.colorbar(cb,ax=ax[1],location='bottom')
ax[1].set_title('Distance Transform of Matrix')
for a in ax:
  a.set_xlim(0,100); a.set_ylim(0,100)

### Finding Centers of Objects with Distance Transforms
An important application of distance transforms is finding centers of objects. We will do this at a smaller scale for clarity.

In [ ]:
small_smoothed = smoothed[:200,:200]
new_bin = small_smoothed < history[-1]
labeled_array, num_labels = ndi.label(new_bin)
fig, ax = plt.subplots(1,2)
cb = ax[0].imshow(small_smoothed)
plt.colorbar(cb,ax=ax[0],location='bottom',label='Height (nm)')
cb = ax[1].imshow(labeled_array)
plt.colorbar(cb,ax=ax[1],location='bottom',label='Region number')
ax[0].set_title('Image'); ax[1].set_title('Identified domains')

The above image on the right is kind of cool, in my opinion; each identified domain corresponds to a connected region. They are color coded to correspond to the number for each region.


The next block calculates the distance transform and uses it to find the centers.

In [ ]:
edt_result = ndi.distance_transform_edt(new_bin)
centers = ndi.maximum_position(edt_result, labels=labeled_array, index=range(1, num_labels + 1))
radii = ndi.maximum(edt_result, labels=labeled_array, index=range(1, num_labels + 1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(labeled_array, cmap='tab10')
axes[0].set_title('Labeled Domains')

axes[1].imshow(edt, cmap='hot')
axes[1].set_title('Distance Transform')

axes[2].imshow(small_smoothed, cmap='gray')
for (row, col), radius in zip(centers, radii):
    axes[2].plot(col, row, 'r+', markersize=12, markeredgewidth=2)
    #circle = plt.Circle((col, row), radius, color='red', fill=False, linewidth=1.5)
    #axes[2].add_patch(circle)
axes[2].set_title('Centers')
axes[2].plot(130,45,'*',markersize=9,color='limegreen')
axes[2].plot(48,70,'*',markersize=9,color='limegreen')
axes[2].plot(37,105,'*', markersize=9,color='limegreen')

plt.tight_layout()
plt.show()

Note that a few domains from the original image were not found because they merged with others during thresholding; in the later section on erosion and dilation, we will explore how to fix that.

## Edge detection & the Image Gradient

In this next part, we will look at how you can find edges in an image. Note that the true image processing names for this are a Sobel filter and a Laplace filter. We will simply use the gradient conceptually. **Know how the 1st and the 2nd derivative of the image gradient work to find edges**.

Images of steel are pulled from the following open-source dataset: https://doi.org/10.6084/m9.figshare.13135625

In [ ]:
steel1 = cv2.imread('/content/MSE7530/sampledata/IMG_00029.png',0)
steel2 = cv2.imread('/content/MSE7530/sampledata/IMG_00248.tif',0)

In [ ]:
fig, ax = plt.subplots(1,2)
cb = ax[0].imshow(steel1,cmap='gray')
plt.colorbar(cb, ax=ax[0],location='bottom')
cb = ax[1].imshow(steel2,cmap='gray')
plt.colorbar(cb, ax=ax[1],location='bottom')

An image-processing goal for this dataset might be to quantify the shapes or grain sizes and connect it to the mechanical properties. We will briefly see how an edge-detection algorithm works. **You should know how a 1st and 2nd gradient of an image can be used to find edges**. These principles are used in Sobel and Laplace filters, and "Canny edge detection"; however, since you are not responsible for this information, I am going to do the edge detection with the Numpy gradient.

First you should pre-process the image to get rid of noise; we will use the SciPy Gaussian filter for that.

In [ ]:
s1_smoothed = ndi.gaussian_filter(steel1, sigma=2.0)
s2_smoothed = ndi.gaussian_filter(steel2, sigma=2.0)

fig, ax = plt.subplots(1,2)
cb = ax[0].imshow(s1_smoothed,cmap='gray')
plt.colorbar(cb, ax=ax[0],location='bottom')
cb = ax[1].imshow(s2_smoothed,cmap='gray')
plt.colorbar(cb, ax=ax[1],location='bottom')

If we take the gradient of the image (the 1st derivative, computed by finite differences - see lecture), we can extract both the locations of the edges and the angle that each of those edges is relative to the image.

In [ ]:
newgrad = np.gradient(s1_smoothed)
np.shape(newgrad)
fig, ax = plt.subplots(1,3,figsize=(15,6))
cb = ax[0].imshow(newgrad[0],cmap='gray')
plt.colorbar(cb,ax=ax[0],location='bottom')
cb = ax[1].imshow(newgrad[1],cmap='gray')
plt.colorbar(cb,ax=ax[1],location='bottom')
cb = ax[2].imshow(np.sqrt(newgrad[0]**2 + newgrad[1]**2),cmap='gray')
plt.colorbar(cb,ax=ax[2],location='bottom')
ax[0].set_title('X-component of image gradient')
ax[1].set_title('Y-component of image gradient')
ax[2].set_title('Magnitude of image gradient')

In [ ]:
orientation = np.arctan2(newgrad[0], newgrad[1])
magnitude   = np.hypot(newgrad[0], newgrad[1])

fig, ax = plt.subplots(1,2,figsize=(10,6))
cb = ax[0].imshow(orientation,cmap='viridis')
plt.colorbar(cb,ax=ax[0],location='bottom')
cb = ax[1].imshow(magnitude,cmap='gray')
plt.colorbar(cb,ax=ax[1],location='bottom')
ax[0].set_title('Angle of Edge (radians)')
ax[1].set_title('Magnitude of image gradient (repeat from above)')

All sorts of things could be calculated with this. For example, one may want to determine the dominant orientation of the grains, or how large a single grain is. What types of quantifications could you do?

## Erosion and Dilation

Erosion and dilation can be used for tasks including disconnecting objects. Take an example from above, where we had some connected holes that threw off center detection.

In [ ]:
new_bin = small_smoothed < history[-1]
labeled_array, num_labels = ndi.label(new_bin)
plt.imshow(labeled_array)
plt.colorbar()


In [ ]:
edt_result = ndi.distance_transform_edt(new_bin)

In [ ]:
centers = ndi.maximum_position(edt_result, labels=labeled_array, index=range(1, num_labels + 1))
radii = ndi.maximum(edt_result, labels=labeled_array, index=range(1, num_labels + 1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(labeled_array, cmap='tab10')
axes[0].set_title('Labeled Domains')

axes[1].imshow(edt, cmap='hot')
axes[1].set_title('Distance Transform')

axes[2].imshow(small_smoothed, cmap='gray')
for (row, col), radius in zip(centers, radii):
    axes[2].plot(col, row, 'r+', markersize=12, markeredgewidth=2)
    #circle = plt.Circle((col, row), radius, color='red', fill=False, linewidth=1.5)
    #axes[2].add_patch(circle)
axes[2].set_title('Centers')
axes[2].plot(130,45,'*',markersize=9,color='limegreen')
axes[2].plot(48,70,'*',markersize=9,color='limegreen')
axes[2].plot(37,105,'*', markersize=9,color='limegreen')

plt.tight_layout()
plt.show()

In [ ]:
imp_binary = small_smoothed < history[-1]

horiz = np.ones((1,3))
vert = np.ones((3,1))

eroded = ndi.binary_erosion(imp_binary,iterations=2, structure=vert) # takes us to the end of where we were at the last loop by specifying iterations
dilated = ndi.binary_dilation(eroded,structure=vert,iterations=2)
fig, ax = plt.subplots(1,3,figsize=(12,4))
ax[0].imshow(imp_binary)
ax[1].imshow(eroded)
ax[2].imshow(dilated)

In [ ]:
labeled_array, num_labels = ndi.label(dilated)
edt_result = ndi.distance_transform_edt(dilated)
centers = ndi.maximum_position(edt_result, labels=labeled_array, index=range(1, num_labels + 1))
radii = ndi.maximum(edt_result, labels=labeled_array, index=range(1, num_labels + 1))
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(labeled_array, cmap='tab10')
axes[0].set_title('Labeled Domains')

axes[1].imshow(edt, cmap='hot')
axes[1].set_title('Distance Transform')

axes[2].imshow(small_smoothed, cmap='gray')
for (row, col), radius in zip(centers, radii):
    axes[2].plot(col, row, 'r+', markersize=12, markeredgewidth=2)
    #circle = plt.Circle((col, row), radius, color='red', fill=False, linewidth=1.5)
    #axes[2].add_patch(circle)
axes[2].set_title('Centers')
axes[2].plot(130,45,'*',markersize=9,color='limegreen')
axes[2].plot(48,70,'*',markersize=9,color='limegreen')
axes[2].plot(37,105,'*', markersize=9,color='limegreen')

plt.tight_layout()
plt.show()



One thing I want to check is how similar in area the new dilation is to the original binary image. I'm going to calculate the percentage of the area of the image that is domain for each.

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(15,5))
ax[0].imshow(imp_binary)
ax[1].imshow(dilated)
ax[0].set_title(f'Original binary: {100*np.sum(imp_binary) / imp_binary.shape[0]**2} % domain')
ax[1].set_title(f'Dilated: {100*np.sum(dilated) / dilated.shape[0]**2} % domain')


What do you notice about the shapes of the different domains? In which situations would this domain change be acceptable? In which would it not?

## Experiment with the shape of the structuring element.  
**You should be able to explain how the effect of choice of structuring element affects the domain separation.** You may choose to use a different portion of the image to test your hypotheses. I will leave slightly more detailed instructions for this soon.

In [ ]:
# Your code here


In [ ]:
# Your code here


## Putting it all together - Counting spherulites in a PVDF (polyvinylidene fluoride) polymer

Read the following documents for information about using watershed algorithms to count image regions:
https://docs.opencv.org/4.13.0/d3/db4/tutorial_py_watershed.html .  
You now have all of the image processing tools you would need, combined with the above tutorial, to perform this for the following image. An example solution will be posted later.

In [ ]:
spher = cv2.imread('/content/MSE7530/sampledata/spherulites_cropped.jpg',0)
plt.imshow(spher)

Crop to a usable region

In [ ]:
cropped = spher[50:500,20:(20+450)]
plt.imshow(cropped)
plt.colorbar()

### ➡️ From here, conduct your own investigation.